# Name That LLM — сборка датасета v1

Тонкий ноутбук-обёртка: запускает `scripts/build_dataset.py` и показывает
`dataset/build/checks.json`. Сами данные и отчёт генерируются скриптом;
ноутбук не хранит больших выходов.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent

cmd = [sys.executable, str(root / "scripts" / "build_dataset.py")]
print(" ".join(cmd))
result = subprocess.run(cmd, cwd=root, check=True, text=True, capture_output=True)
print(result.stdout[-4000:])
if result.stderr:
    print("stderr:", result.stderr[-2000:])


In [ ]:
checks = json.loads((root / "dataset" / "build" / "checks.json").read_text())
print("status:", checks["status"])
for key, value in checks["counts"].items():
    print(f"{key}: {value}")
print("\nfold sizes:", checks["pool_split"]["fold_sizes"])
print("block sizes:", checks["official_split"]["block_sizes"])
print("warnings:", checks["validation"]["warnings"])


In [ ]:
import pandas as pd

build = root / "dataset" / "build"
folds = pd.read_parquet(build / "pool_split_v1.parquet")
display(folds.groupby("fold").agg(
    prompts=("prompt_canon", "size"),
    rows=("n_clean", "sum"),
    models=("n_models", "max"),
    sources=("n_sources", "max"),
))


## Что дальше

- `scripts/validate_dataset.py` — независимая проверка файлов и хэшей.
- `scripts/upload_dataset.py --public` — загрузка на Kaggle, проверка
  скачиванием и публикация.
- `reports/build_dataset_v1.md` — сгенерированный отчёт.
